# 03 · Fusion and temporal aggregation (Step 4)

Fuse the frame curve **s(t)** (SigLIP, notebook 01) with the activity curve **a(t)** (X-CLIP, notebook 02):

**f(t) = α · z(s) + (1 − α) · z(a)**

α = 1 is frame-only and α = 0 is activity-only, so every row of the ablation table comes from the same code with the same tuning budget. Everything is tuned on **val** (693 queries) and evaluated **once** on **test** (3,720 queries).

**Session settings:** Accelerator **None** (CPU only, no GPU quota) · Internet **On**
**Inputs (Add Input → Datasets):** `charades-sta-vidsearch` (annotations), `vidsearch-features` (SigLIP), `vidsearch-activity` (X-CLIP).

**Workflow:** click **Run All** (about 5 minutes), check the outputs, then *Save Version → Save & Run All (Commit)* to keep the results.

In [ ]:
# 1. Get the code from GitHub and run every unit test
from pathlib import Path
import sys, glob, subprocess

REPO_URL = "https://github.com/zankhana46/AI-Video-Investigator.git"
!rm -rf /tmp/repo && git clone -q $REPO_URL /tmp/repo
sys.path.insert(0, "/tmp/repo")

for t in sorted(glob.glob("/tmp/repo/tests/test_*.py")):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, cwd="/tmp/repo")
    print(r.stdout.strip() or f"(no output from {Path(t).name})")
    if r.returncode:
        print(r.stderr[-2000:])
assert Path("/tmp/repo/vidsearch/fusion.py").exists(), "Step 4 files are not on GitHub yet: push them first"

In [ ]:
# 2. Find the three attached datasets
import json
from pathlib import Path
IN = Path("/kaggle/input")

ann = sorted(IN.rglob("val.jsonl"))
assert ann, "Attach charades-sta-vidsearch (Add Input -> Datasets)"
ANN_DIR = ann[0].parent

siglip = [p for p in IN.rglob("*.npz") if not p.name.startswith("xclip_")]
assert siglip, "Attach vidsearch-features (SigLIP features from notebook 01)"
FRAME_FEAT_DIR = siglip[0].parent

meta = sorted(IN.rglob("xclip_meta.json"))
assert meta, "Attach vidsearch-activity (X-CLIP features from notebook 02)"
ACT_DIR = meta[0].parent

s3 = sorted(IN.rglob("step3_summary.json"))
ACT_VARIANT = json.load(open(s3[0]))["activity_variant"].split("_")[-1] if s3 else "plain"

print("annotations:      ", ANN_DIR)
print("SigLIP features:  ", FRAME_FEAT_DIR)
print("X-CLIP features:  ", ACT_DIR)
print("activity variant: ", ACT_VARIANT, "(chosen on val in Step 3)")

In [ ]:
# 3. Settings
import numpy as np
ALPHAS = np.round(np.arange(0, 1.0001, 0.05), 2)     # weight on the frame curve; 1 = frame only, 0 = activity only
FUSED_ALPHAS = [a for a in ALPHAS if 0 < a < 1]      # what counts as "fused"
N_JOBS = None                                        # None = use every CPU core
BOOT = 5000                                          # bootstrap resamples for the confidence intervals

WORK = Path("/kaggle/working")
RES_OUT = WORK / "results"
RES_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RES_OUT / "results_step4.csv"

# Reference rows printed by notebook 00 on the full test set
BASELINES = [{"evidence": "random", "method": "-", "R1@0.5": 11.7, "R1@0.7": 2.7, "R5@0.5": 37.6, "mIoU": 23.7},
             {"evidence": "prior", "method": "-", "R1@0.5": 13.8, "R1@0.7": 4.5, "R5@0.5": 13.8, "mIoU": 17.4}]

In [ ]:
# 4. Load annotations and both curves for every query
from vidsearch import curves as cv
from vidsearch import fusion as fu
from vidsearch.xclip_features import load_activity, activity_curves

val = cv.load_jsonl(ANN_DIR / "val.jsonl", "val")
test = cv.load_jsonl(ANN_DIR / "test.jsonl", "test")
recs = val + test

S = cv.load_frame_curves(FRAME_FEAT_DIR, recs)
assert S, "SigLIP feature layout not recognised: send Claude the lines above"
act = load_activity(ACT_DIR)
A = activity_curves(act, recs, ACT_VARIANT)
print("activity model:", act["config"]["model"])

for name, R in (("val", val), ("test", test)):
    both = sum(r["qid"] in S and r["qid"] in A for r in R)
    print(f"{name}: {len(R)} queries, {both} with both curves")
    assert both >= 0.99 * len(R), "curves missing for some queries"

print("sanity (Step 3 found 76.5 and 68.0):",
      "frame", cv.inside_vs_outside(S, val)[0], "| activity", cv.inside_vs_outside(A, val)[0])

In [ ]:
# 5. Sweep alpha x proposal settings on VAL (CPU, a few minutes)
import time
t0 = time.time()
VAL_ROWS = fu.sweep(S, A, val, ALPHAS, n_jobs=N_JOBS)
print(f"{len(VAL_ROWS)} configurations evaluated on {len(val)} val queries in {time.time() - t0:.0f} s")

In [ ]:
# 6. Pick each row's settings on VAL: frame (alpha = 1), activity (alpha = 0), fused (0 < alpha < 1)
#    hybrid = threshold's top-1 (best boundaries) + windows for ranks 2-5 (best candidate sets)
import pandas as pd

EVIDENCE = {"frame": [1.0], "activity": [0.0], "fused": FUSED_ALPHAS}
CHOSEN, VAL_TABLE = {}, []
for ev, al in EVIDENCE.items():
    thr = fu.best_row(VAL_ROWS, "threshold", al)
    win = fu.best_row(VAL_ROWS, "windows", al)
    rest, hyb = fu.best_hybrid(VAL_ROWS, thr, val, al)
    CHOSEN[(ev, "threshold")] = {"top": (thr["alpha"], "threshold", thr["params"])}
    CHOSEN[(ev, "windows")] = {"top": (win["alpha"], "windows", win["params"])}
    CHOSEN[(ev, "hybrid")] = {"top": (thr["alpha"], "threshold", thr["params"]),
                              "rest": (rest["alpha"], "windows", rest["params"])}
    for meth, m in (("threshold", thr), ("windows", win), ("hybrid", hyb)):
        c = CHOSEN[(ev, meth)]
        VAL_TABLE.append({"evidence": ev, "method": meth, "alpha": c["top"][0],
                          "alpha (ranks 2-5)": c["rest"][0] if "rest" in c else "",
                          **{k: m[k] for k in ("R1@0.5", "R1@0.7", "R5@0.5", "mIoU")}})
display(pd.DataFrame(VAL_TABLE))
pd.DataFrame([{k: v for k, v in r.items() if k != "P"} for r in VAL_ROWS]).to_csv(RES_OUT / "val_sweep.csv", index=False)

In [ ]:
# 7. How does accuracy change with alpha? (VAL, best settings per alpha)
import matplotlib.pyplot as plt
tab = pd.DataFrame(fu.alpha_table(VAL_ROWS))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ("R1@0.5", "mIoU")):
    for meth, col in (("threshold", "tab:green"), ("windows", "tab:purple")):
        d = tab[tab.method == meth]
        ax.plot(d.alpha, d[metric], "o-", color=col, label=meth)
    best_a = CHOSEN[("fused", "threshold")]["top"][0]
    ax.axvline(best_a, color="gray", ls="--", lw=1, label=f"chosen alpha = {best_a}")
    ax.set_xlabel("alpha  (0 = activity only, 1 = frame only)"); ax.set_ylabel(f"val {metric}")
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout(); plt.savefig(RES_OUT / "alpha_sweep_val.png", dpi=130); plt.show()

In [ ]:
# 8. TEST, once, with the frozen val settings
_FT = {}
def curves_at(alpha):                               # fused test curves, cached per alpha
    if alpha not in _FT:
        _FT[alpha] = fu.fused_curves(S, A, test, alpha)
    return _FT[alpha]

def run(spec):
    alpha, method, params = spec
    return fu.predict_array(curves_at(alpha), test, method, params)

TEST_ROWS, PER = [], {}
for (ev, meth), c in CHOSEN.items():
    P = run(c["top"])
    if "rest" in c:
        P = fu.hybrid(P, run(c["rest"]))
    m, per = fu.evaluate_array(P, test)
    PER[(ev, meth)] = {"P": P, **per}
    TEST_ROWS.append({"evidence": ev, "method": meth, **m})
    cv.log_row(RESULTS_CSV, f"{ev}_{meth}", "test", m, c)

cols = ["evidence", "method", "R1@0.3", "R1@0.5", "R1@0.7", "R5@0.5", "R5@0.7", "mIoU", "n_queries"]
table = pd.DataFrame(BASELINES + TEST_ROWS).reindex(columns=cols)
table["n_queries"] = table["n_queries"].fillna(len(test)).astype(int)    # baselines used the full test set
display(table.fillna("–"))

In [ ]:
# 9. Is the gain real? 95% confidence intervals for fused minus frame (and minus activity),
#    resampling whole test videos (queries of the same video are not independent)
groups = [r["vid"] for r in test]
SIG = []
for meth in ("threshold", "windows", "hybrid"):
    f = PER[("fused", meth)]
    for other in ("frame", "activity"):
        o = PER[(other, meth)]
        for metric, x, y in (("R1@0.5", f["top1"] >= 0.5, o["top1"] >= 0.5),
                             ("R5@0.5", f["best5"] >= 0.5, o["best5"] >= 0.5),
                             ("mIoU", f["top1"], o["top1"])):
            SIG.append({"method": meth, "comparison": f"fused - {other}", "metric": metric,
                        **fu.cluster_bootstrap(x, y, groups, B=BOOT)})
SIG = pd.DataFrame(SIG)
SIG["significant"] = (SIG.ci_low > 0) | (SIG.ci_high < 0)
display(SIG)
SIG.to_csv(RES_OUT / "significance_test.csv", index=False)
print("diff = points gained by fusion; significant = the 95% CI excludes 0")

In [ ]:
# 10. Challenging cases (TEST): R1@0.5 by moment length, and on opposite-action queries
cat = np.array([fu.duration_category(r) for r in test])
opp = {q for _, a, b in cv.find_opposite_pairs(test) for q in (a["qid"], b["qid"])}
groups_cat = {c: cat == c for c in ("brief (<3 s)", "medium", "long (>10 s or >30%)")}
groups_cat["opposite-action queries"] = np.array([r["qid"] in opp for r in test])

CAT = []
for name, mask in groups_cat.items():
    row = {"category": name, "n": int(mask.sum())}
    for ev in ("frame", "activity", "fused"):
        top1 = PER[(ev, "hybrid")]["top1"][mask]
        row[f"{ev} R1@0.5"] = round(100 * float((top1 >= 0.5).mean()), 1) if mask.any() else float("nan")
    CAT.append(row)
CAT = pd.DataFrame(CAT)
display(CAT)
CAT.to_csv(RES_OUT / "categories_test.csv", index=False)

In [ ]:
# 11. Where fusion fixes or breaks a query (TEST): s(t), a(t), f(t), GT and each top-1 prediction
import textwrap
fr, fu_ = PER[("frame", "hybrid")], PER[("fused", "hybrid")]
fixes = np.flatnonzero((fu_["top1"] >= 0.5) & (fr["top1"] < 0.3))
hurts = np.flatnonzero((fu_["top1"] < 0.3) & (fr["top1"] >= 0.5))
print(f"fusion fixed {len(fixes)} test queries and broke {len(hurts)} (top-1 IoU >= 0.5 vs < 0.3)")

rng = np.random.default_rng(0)
pick = [(i, "FIXED") for i in rng.choice(fixes, min(3, len(fixes)), replace=False)] + \
       [(i, "BROKE") for i in rng.choice(hurts, min(1, len(hurts)), replace=False)]
a_top = CHOSEN[("fused", "hybrid")]["top"][0]
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, (i, tag) in zip(axes.flat, pick):
    r = test[i]; q = r["qid"]
    s, a = cv.align(S[q], A[q]); f = curves_at(a_top)[q]; t = np.arange(len(f)) + 0.5
    ax.plot(t, cv.zscore(s), color="tab:blue", alpha=0.7, label="frame s(t)")
    ax.plot(t, cv.zscore(a), color="tab:orange", alpha=0.7, label="activity a(t)")
    ax.plot(t, cv.zscore(f), color="black", lw=2, label=f"fused f(t), alpha={a_top}")
    ax.axvspan(r["start"], r["end"], color="green", alpha=0.2, label="GT")
    top = ax.get_ylim()[1]
    ax.hlines(top * 0.95, *fr["P"][i, 0], color="tab:blue", lw=6, label="frame top-1")
    ax.hlines(top * 0.80, *fu_["P"][i, 0], color="black", lw=6, label="fused top-1")
    ax.set_title(f"{tag}: " + textwrap.shorten(r["query"], 55) +
                 f"\nIoU frame {fr['top1'][i]:.2f} -> fused {fu_['top1'][i]:.2f}", fontsize=9)
    ax.set_xlabel("seconds")
if pick:
    axes.flat[0].legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.savefig(RES_OUT / "fusion_examples_test.png", dpi=130); plt.show()

In [ ]:
# 12. Save the summary, the final configuration (for the demo) and a report-ready table
c = CHOSEN[("fused", "hybrid")]
FINAL = {"alpha_top1": c["top"][0], "threshold_params": c["top"][2],
         "alpha_ranks_2_5": c["rest"][0], "windows_params": c["rest"][2],
         "activity_model": act["config"]["model"], "activity_variant": ACT_VARIANT,
         "frame_model": "SigLIP (notebook 01)", "clip_sec": act["config"]["clip_sec"]}
json.dump({"final_config": FINAL,
           "chosen": {f"{e}_{m}": v for (e, m), v in CHOSEN.items()},
           "test": TEST_ROWS, "val": VAL_TABLE, "significance": SIG.to_dict("records"),
           "categories": CAT.to_dict("records"), "fixed": int(len(fixes)), "broke": int(len(hurts))},
          open(RES_OUT / "step4_summary.json", "w"), indent=2, default=str)
json.dump({f"{e}_{m}": dict(zip([r["qid"] for r in test], map(float, v["top1"])))
           for (e, m), v in PER.items()}, open(RES_OUT / "per_query_top1_iou_step4.json", "w"))

AGG = {"threshold": "threshold + merge", "windows": "multi-scale windows", "hybrid": "hybrid"}
lines = ["| Evidence | Temporal aggregation | R1@0.5 | R1@0.7 | R5@0.5 | R5@0.7 | mIoU |",
         "|---|---|---|---|---|---|---|"]
for b in BASELINES:
    lines.append(f"| {b['evidence']} | – | {b['R1@0.5']} | {b['R1@0.7']} | {b['R5@0.5']} | – | {b['mIoU']} |")
for r in TEST_ROWS:
    ours = r["evidence"] == "fused" and r["method"] == "hybrid"
    cells_ = [r[k] for k in ("R1@0.5", "R1@0.7", "R5@0.5", "R5@0.7", "mIoU")]
    cells_ = [f"**{v}**" if ours else str(v) for v in cells_]
    lines.append(f"| {r['evidence']}{' (ours)' if ours else ''} | {AGG[r['method']]} | " + " | ".join(cells_) + " |")
(RES_OUT / "ablation_table.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
print("\nfinal configuration:", FINAL)
!ls {RES_OUT}

### Save the results
1. **Save Version → Save & Run All (Commit)** (Accelerator stays **None**, so it uses no GPU quota).
2. When it finishes: **Output → New Dataset**, name it `vidsearch-fusion`, and share it with your teammate. It holds the final configuration that the demo (Step 7) and the challenging-case analysis (Step 6) will reuse.